# MultiAmr2Cube — Sub-cube benchmark

Compares three variants on full box and power-of-2 aligned sub-cubes:

| Variant | Pre-filter | Kernel |
|---|---|---|
| `ref` | none | numpy loop |
| `ref_pf` | spatial pre-filter | numpy loop |
| `numba_full_pf` | spatial pre-filter | fully-numba single-pass |

Sub-cubes tested (power-of-2 aligned, lower corner at origin):
- full box  `[0,1,0,1,0,1]`  → f=1
- half box  `[0,.5,0,.5,0,.5]` → f=1/8 of volume
- quarter box `[0,.25,0,.25,0,.25]` → f=1/64 of volume

**Run from `multiamr2cube_optim/`.**

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import time
from configparser import ConfigParser

BASE_DIR   = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'claude' else os.getcwd()
CLAUDE_DIR = os.path.join(BASE_DIR, 'claude')
sys.path.insert(0, BASE_DIR)
sys.path.insert(0, CLAUDE_DIR)

import pyablo
import multiamr2cube_ref                as m_ref
import multiamr2cube_ref_prefilter      as m_ref_pf
import multiamr2cube_numba_full_prefilter as m_numba_full_pf

print('BASE_DIR:', BASE_DIR)

## 1 — Load snapshot (once)

In [ ]:
PATH       = BASE_DIR + '/'
SNAPNAME   = 'star'
CONFIGNAME = 'restart_star_279982.ini'
T          = 70000
FIELD      = 'rho'

config = ConfigParser(inline_comment_prefixes=('#', ';'))
config.read(PATH + CONFIGNAME)

reader = pyablo.XdmfReader()
snap   = reader.readSnapshot(PATH + SNAPNAME + '_iter{:07d}.xmf'.format(T))

cc, lcell, Lbox = m_ref.extract_snapshot_data(snap, config)
values = np.array(snap.readAllFloat(FIELD), dtype=np.float64)

lmin = int(lcell.min())
lmax = int(lcell.max())
print(f'{len(values):,} cells | lmin={lmin} lmax={lmax} | Lbox={Lbox}')

## 2 — Warm up numba

In [ ]:
print('Compiling numba kernel...')
t0 = time.perf_counter()
m_numba_full_pf.warmup(cc, lcell, values, Lbox)
print(f'  done in {time.perf_counter()-t0:.1f}s')

## 3 — Pre-filter statistics
How many cells survive for each sub-cube?

In [ ]:
EXTENTS = {
    'full  (f=1)   ': [0, 1,    0, 1,    0, 1   ],
    'half  (f=1/8) ': [0, 0.5,  0, 0.5,  0, 0.5 ],
    'quarter(f=1/64)': [0, 0.25, 0, 0.25, 0, 0.25],
}

print(f'Total cells: {len(values):,}')
print()
for label, extent in EXTENTS.items():
    cc_f, lc_f, v_f = m_ref_pf.prefilter(cc, lcell, values, Lbox, extent)
    frac = len(v_f) / len(values)
    print(f'  {label}  →  {len(v_f):>10,} cells  ({frac*100:.1f}% of total)')

## 4 — Benchmark

In [ ]:
LCUBES = [8, 9]
N_REPEATS = 5

VARIANTS = {
    'ref':           m_ref.multiamr2cube,
    'ref_pf':        m_ref_pf.multiamr2cube,
    'numba_full_pf': m_numba_full_pf.multiamr2cube,
}

results = {}   # (name, lcube, extent_label) -> {'times': [...], 'result': array}

for lcube in LCUBES:
    for label, extent in EXTENTS.items():
        print(f'\n=== lcube={lcube}  extent={label.strip()} ===')
        ref_arr = None
        for name, fn in VARIANTS.items():
            times = []
            for rep in range(N_REPEATS):
                t0 = time.perf_counter()
                arr = fn(cc, lcell, values, Lbox, lcube, extent)
                times.append(time.perf_counter() - t0)
            key = (name, lcube, label)
            results[key] = {'times': times, 'result': arr}
            t_med = np.median(times)
            if ref_arr is None:
                ref_arr = arr
                correct = 'REF'
            else:
                rel = np.max(np.abs(arr - ref_arr)) / (np.max(np.abs(ref_arr)) + 1e-300)
                correct = f'OK (rel={rel:.1e})' if rel < 1e-10 else f'FAIL rel={rel:.2e}'
            print(f'  {name:<16}  median={t_med:.3f}s   {correct}')

## 5 — Timing plot

In [ ]:
extent_labels = list(EXTENTS.keys())
variant_names = list(VARIANTS.keys())
colors = {'ref': 'steelblue', 'ref_pf': 'darkorange', 'numba_full_pf': 'crimson'}

fig, axes = plt.subplots(len(LCUBES), len(extent_labels),
                          figsize=(5 * len(extent_labels), 4 * len(LCUBES)),
                          sharey='row')

for row, lcube in enumerate(LCUBES):
    for col, label in enumerate(extent_labels):
        ax = axes[row][col]
        medians = [np.median(results[(n, lcube, label)]['times']) for n in variant_names]
        mins    = [np.min(   results[(n, lcube, label)]['times']) for n in variant_names]
        bars = ax.bar(variant_names, medians,
                      color=[colors[n] for n in variant_names], alpha=0.85)
        ax.scatter(variant_names, mins, marker='_', s=200, color='black', zorder=5)
        ref_med = np.median(results[('ref', lcube, label)]['times'])
        for bar, med in zip(bars, medians):
            ax.text(bar.get_x() + bar.get_width()/2, med * 1.02,
                    f'{ref_med/med:.1f}x', ha='center', va='bottom', fontsize=9)
        ax.set_title(f'lcube={lcube}  {label.strip()}', fontsize=10)
        ax.set_ylabel('Wall time (s)' if col == 0 else '')
        ax.tick_params(axis='x', rotation=20)

plt.suptitle('Sub-cube benchmark\n(black tick = min, number = speedup vs ref)',
             fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(CLAUDE_DIR, 'timing_subcube.png'), dpi=150)
plt.show()
print('Saved timing_subcube.png')

## 6 — Pre-filter scaling: time vs sub-cube volume fraction

In [ ]:
lcube_plot = LCUBES[0]
fracs = [1.0, 1/8, 1/64]

fig, ax = plt.subplots(figsize=(6, 4))
for name in variant_names:
    times_by_frac = [np.median(results[(name, lcube_plot, lbl)]['times'])
                     for lbl in extent_labels]
    ax.plot(fracs, times_by_frac, 'o-', label=name, color=colors[name])

# ideal 1/f scaling line anchored at ref full-box time
t_ref_full = np.median(results[('ref', lcube_plot, extent_labels[0])]['times'])
f_fine = np.logspace(-2, 0, 50)
ax.plot(f_fine, t_ref_full * f_fine, 'k--', alpha=0.4, label='ideal 1/f')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Sub-cube volume fraction f')
ax.set_ylabel('Wall time (s)')
ax.set_title(f'lcube={lcube_plot} — time vs sub-cube size')
ax.legend()
ax.invert_xaxis()
plt.tight_layout()
plt.savefig(os.path.join(CLAUDE_DIR, 'scaling_subcube.png'), dpi=150)
plt.show()
print('Saved scaling_subcube.png')